<div align="right"><sub>Notebook 最終更新: </sub></div>
<h1><strong>08. RAG 統合マルチエージェント（最終回）</strong></h1>

演習の締めくくりとして，これまでに学んだ **RAG (外部知識の参照)** と **AIエージェント (Executor & Critic)** を統合します。

LLMが本来持っていない架空の知識（2027年のアニメニュース）について回答し、その内容が検索結果と矛盾していないか、資料にない嘘（ハルシネーション）をついていないかを Critic が厳密に検証する、高度な信頼性を持ったシステムを構築しましょう。

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu peft datasets gradio

import os
import sys
from google.colab import drive

DRIVE_MOUNT_POINT = '/content/drive'
drive.mount(DRIVE_MOUNT_POINT, force_remount=False)

PERSIST_ROOT = os.path.join(DRIVE_MOUNT_POINT, 'MyDrive', 'AIAgent')
PERSIST_INDEX_DIR = os.path.join(PERSIST_ROOT, 'data', 'index')
os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)

REPO_ROOT = '/content/llm_lab'
if not os.path.exists(REPO_ROOT):
    !git clone -b ai_agent https://github.com/akio-kobayashi/llm_lab.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

os.chdir(REPO_ROOT)
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

print('現在の作業ディレクトリ:', os.getcwd())
from src.common import load_llm, generate_text, AGENT_MODEL_ID
from src.rag import RagEngine
from src.agent_core import LLMExecutorCriticAgent, RoleConfig
from src.ui import create_agent_ui

model, tokenizer = load_llm(model_id=AGENT_MODEL_ID)
print('準備完了')


## **1. RAG統合エージェントの設定**
Executor には「RAG情報を元に回答する（Fact Finder）」役割を、Critic には「回答がRAG情報と矛盾していないか評価する（Fact Checker）」役割を与えます。

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 768, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

# RAGの初期化
rag = RagEngine()
index_path = os.path.join(PERSIST_INDEX_DIR, 'faiss.index')
chunks_path = os.path.join(PERSIST_INDEX_DIR, 'chunks.json')

if os.path.exists(index_path) and os.path.exists(chunks_path):
    rag.load_index(index_path, chunks_path)
else:
    rag.load_documents('data/docs/anime_docs_sample.jsonl')
    rag.build_index()
    os.makedirs(PERSIST_INDEX_DIR, exist_ok=True)
    rag.save_index(index_path, chunks_path)

executor_prompt = """
あなたは優秀な調査官（Fact Finder）です。
提供された【資料】の内容だけに基づいて、質問に正確に回答してください。
資料に記載されていない情報については、絶対に推測で答えず「資料には記載されていません」と明記してください。
"""

critic_prompt = """
あなたは厳格なファクトチェッカーです。
提出された回答が、提供された【資料】と完全に一致しているか、以下の基準で評価してください：
1. 回答内容が資料と矛盾していないか
2. 資料に存在しない情報（ハルシネーション）を勝手に捏造して追加していないか

もし1つでも違反があれば、絶対に自分で修正せず、違反箇所を指摘して書き直しを求めてください。
提出された回答が事実と完全に一致している場合のみ、「誤りなし」と出力して承認してください。
"""

executor_config = RoleConfig(name='Fact Finder', system_prompt=executor_prompt)
critic_config = RoleConfig(name='Fact Checker', system_prompt=critic_prompt)

agent = LLMExecutorCriticAgent(llm_chat, role_configs=[executor_config, critic_config])
print('エージェントとRAGの準備が完了しました。')


## **2. 統合UIの起動**
質問を入力すると、「RAG検索 → Fact Finderの初稿 → Fact Checkerの検証 → 修正（必要な場合）」のフルプロセスが実行されます。

### 試してみるクエリの例（コピペ用）
1. **資料に基づいた正確な事実確認**:
   > 「2027年公開予定のSFアニメ『星空のレクイエム』のあらすじと，舞台となる都市の名前を教えてください。」
2. **（意図的なハルシネーション誘発テスト）**:
   > 「『タイムリープ・カフェ』の主題歌を歌っている人気アーティストは誰ですか？推測でもいいので教えてください。」
3. **複数の情報をまとめる**:
   > 「2027年に公開されるアニメ作品について、資料に登場するすべての作品タイトルを一覧にしてください。」

In [ ]:
def run_rag_agent(query):
    # 1. RAG検索
    results = rag.search(query, top_k=3)
    context = ""
    for i, res in enumerate(results):
        context += f"[資料{i+1}] {res['chunk']['text']}\n"
    
    # 2. エージェントへの入力構築
    agent_input = f"【資料】\n{context}\n\n【質問】\n{query}"
    
    # 3. エージェント実行（最大2回の自己修正ループを許可）
    final_answer, full_log, steps = agent.run_pipeline(agent_input, max_iterations=2)
    
    formatted_log = f"### RAG 検索結果\n"
    for i, res in enumerate(results):
        formatted_log += f"* {res['chunk']['title']}: {res['chunk']['text'][:50]}...\n"
    
    formatted_log += "\n--- AIエージェントの処理過程 ---\n"
    for s in steps:
        formatted_log += f"### {s.role}\n> {s.observation.replace('\n', '\n> ')}\n\n"
        
    return final_answer, formatted_log

ui = create_agent_ui(run_rag_agent)
ui.launch(share=True)

## **演習のまとめ（コース修了）**

このコースでは、日本語対応の強力なローカルLLM **Qwen** シリーズを使って、以下のステップを学びました：

1. **LLMの基礎**: 生成の仕組みとハルシネーションの限界
2. **プロンプト制御**: 指示による振る舞いの変化
3. **RAG (検索拡張生成)**: 外部の膨大な知識をLLMに「読ませる」方法
4. **AIエージェント**: 複数のペルソナ（執筆者と編集長、調査官と監査役など）を組み合わせた**「自律的で信頼性の高いシステム」**の構築

単体ではハルシネーションを起こしたり推敲が甘かったりするLLMも、**RAG** や **反復的なエージェント（Iterative Refinement）** などの仕組みと組み合わせることで、エンタープライズでも通用する極めて高品質・高信頼なAIシステムへと進化させることができます。

このリポジトリでの学びを足がかりにして、ぜひあなただけの高度なオリジナルAIアプリケーション開発に挑戦してみてください！